# Multivariate Normal & LKJ Priors

This notebook demonstrates modeling correlated outcomes with `MultivariateNormal` likelihood.

**Two scenarios:**
1. **Known covariance** - Estimate means only
2. **Unknown covariance** - Estimate means, scales, and correlation using `LKJCholesky` prior

## Example: Bivariate Test Scores

Students take Math and Reading tests. Scores are correlated (ρ ≈ 0.67).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import minibayes as mb
from minibayes import Model, dist, viz

---
## Data

In [ ]:
# True parameters
mu_true = np.array([70.0, 75.0])  # Math=70, Reading=75
cov_true = np.array([[100.0, 60.0],
                     [60.0, 81.0]])  # Correlation = 0.67
rho_true = cov_true[0, 1] / np.sqrt(cov_true[0, 0] * cov_true[1, 1])

# Generate data
rng = np.random.default_rng(42)
y_obs = rng.multivariate_normal(mu_true, cov_true, size=200)

print(f"True: mu_math={mu_true[0]}, mu_reading={mu_true[1]}, ρ={rho_true:.2f}")
print(f"Sample: mu_math={y_obs[:, 0].mean():.1f}, mu_reading={y_obs[:, 1].mean():.1f}")

---
## Part 1: Known Covariance

When the covariance is known (from prior studies), we only estimate the means.

In [ ]:
from numpy.typing import NDArray

cov_known = cov_true

def priors(p):
    p("mu_math", dist.Normal(70, 15))
    p("mu_reading", dist.Normal(70, 15))

def log_likelihood(params, data) -> NDArray[np.float64]:
    mean = np.array([params["mu_math"], params["mu_reading"]])
    return dist.MultivariateNormal(mean=mean, cov=cov_known).log_prob(data)

model = Model(priors=priors, log_likelihood=log_likelihood)
print(f"Parameters: {model.param_names}")

In [ ]:
result = mb.sample(
    model, data=y_obs,
    num_samples=10000, num_warmup=1000,
    num_chains=2, sampler="adaptive_mh", seed=42, progress=True
)

In [ ]:
print(viz.summary_table(result))
fig = viz.plot_samples(result)
plt.show()

In [ ]:
# Posterior densities
fig = viz.plot_density(result)
plt.show()

# Joint posterior - shows correlation structure
fig = viz.plot_pair(result, params=["mu_math", "mu_reading"],
                    markers={"True": (mu_true[0], mu_true[1])})
plt.show()

---
## Part 2: Unknown Covariance with LKJ Prior

When covariance is unknown, use `LKJCholesky` to estimate the correlation.

**Model structure:**
- `mu_math`, `mu_reading` - means (Normal priors)
- `sigma_math`, `sigma_reading` - standard deviations (HalfNormal priors)  
- `L_corr` - Cholesky factor of correlation matrix (LKJ prior)

In [ ]:
def priors_lkj(p):
    p("mu_math", dist.Normal(70, 15))
    p("mu_reading", dist.Normal(70, 15))
    p("sigma_math", dist.HalfNormal(20))
    p("sigma_reading", dist.HalfNormal(20))
    p("L_corr", dist.LKJCholesky(dim=2, eta=2.0), shape=(2, 2))

def log_likelihood_lkj(params, data) -> NDArray[np.float64]:
    mean = np.array([params["mu_math"], params["mu_reading"]])
    # Reconstruct covariance: cov = diag(sigma) @ corr @ diag(sigma)
    L_corr = params["L_corr"]
    corr = L_corr @ L_corr.T
    sigma = np.diag([params["sigma_math"], params["sigma_reading"]])
    cov = sigma @ corr @ sigma
    return dist.MultivariateNormal(mean=mean, cov=cov).log_prob(data)

model_lkj = Model(priors=priors_lkj, log_likelihood=log_likelihood_lkj)
print(f"Parameters: {model_lkj.param_names}")

In [ ]:
result_lkj = mb.sample(
    model_lkj, data=y_obs,
    num_samples=30000, num_warmup=5000,
    num_chains=2, sampler="adaptive_mh", seed=42, progress=True
)

In [ ]:
# Diagnostics for scalar parameters
print(viz.summary_table(result_lkj, params=["mu_math", "mu_reading", "sigma_math", "sigma_reading"]))
fig = viz.plot_samples(result_lkj, params=["mu_math", "mu_reading", "sigma_math", "sigma_reading"])
plt.show()

In [ ]:
# Extract correlation from L_corr (for 2x2: rho = L[1,0])
# Use add_derived() to make it available to viz functions
L_samples = result_lkj.samples["L_corr"]
rho_samples = L_samples[:, :, 1, 0]  # shape (num_chains, num_samples)
result_lkj.add_derived("rho", rho_samples)

# Now we can use summary() directly on the derived parameter
print(viz.summary_table(result_lkj, params=["rho"]))
print(f"\nTrue correlation: {rho_true:.3f}")

In [ ]:
# Visualize all posteriors including derived rho
fig = viz.plot_density(result_lkj, params=["mu_math", "mu_reading", "sigma_math", "sigma_reading", "rho"])
plt.show()

# Can also plot just correlation with true value marker
with viz.style():
    fig, ax = plt.subplots(figsize=(6, 4))
    rho_flat = result_lkj.derived["rho"].flatten()
    ax.hist(rho_flat, bins=50, density=True, alpha=0.7, color=viz.PALETTE["blue"])
    ax.axvline(rho_true, color=viz.PALETTE["terracotta"], lw=2, ls="--", label=f"True={rho_true:.2f}")
    ax.axvline(rho_flat.mean(), color=viz.PALETTE["sage"], lw=2, label=f"Est={rho_flat.mean():.2f}")
    ax.set_xlabel("Correlation (ρ)")
    ax.set_ylabel("Density")
    ax.set_title("Correlation Posterior")
    ax.legend()
    plt.show()

In [ ]:
# Compare results
print("\n=== Parameter Estimates ===")
print(f"{'Parameter':<15} {'True':>8} {'Known Cov':>12} {'LKJ Model':>12}")
print("-" * 50)
print(f"{'mu_math':<15} {mu_true[0]:>8.1f} {result.samples['mu_math'].mean():>12.1f} {result_lkj.samples['mu_math'].mean():>12.1f}")
print(f"{'mu_reading':<15} {mu_true[1]:>8.1f} {result.samples['mu_reading'].mean():>12.1f} {result_lkj.samples['mu_reading'].mean():>12.1f}")
print(f"{'sigma_math':<15} {np.sqrt(cov_true[0,0]):>8.1f} {'(fixed)':>12} {result_lkj.samples['sigma_math'].mean():>12.1f}")
print(f"{'sigma_reading':<15} {np.sqrt(cov_true[1,1]):>8.1f} {'(fixed)':>12} {result_lkj.samples['sigma_reading'].mean():>12.1f}")
print(f"{'correlation':<15} {rho_true:>8.2f} {'(fixed)':>12} {result_lkj.derived['rho'].mean():>12.2f}")

In [ ]:
# Save and load results (derived parameters are preserved)
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmpdir:
    path = Path(tmpdir) / "result_lkj.npz"
    result_lkj.save(str(path))
    loaded = mb.InferenceResult.load(str(path))

    print("Loaded result contains derived parameters:")
    print(f"  Derived params: {list(loaded.derived.keys())}")
    print(f"  rho mean: {loaded.derived['rho'].mean():.3f}")
    print(f"  Original rho mean: {result_lkj.derived['rho'].mean():.3f}")

---
## Key Takeaways

### MultivariateNormal
```python
dist.MultivariateNormal(mean=mean, cov=cov).obs_logp(data)
```
- Use as **likelihood** for correlated observations
- Data shape: `(n, d)` for n observations in d dimensions

### LKJCholesky Prior
```python
L_corr = p("L_corr", dist.LKJCholesky(dim=d, eta=2.0), shape=(d, d))
corr = L_corr @ L_corr.T  # Correlation matrix
cov = diag(sigma) @ corr @ diag(sigma)  # Full covariance
```
- **eta=1**: Uniform over correlations
- **eta>1**: Favors weaker correlations (closer to identity)
- Use `shape=(d,d)` for matrix parameters (not `size`)

### Derived Parameters
```python
# Extract correlation from Cholesky factor
rho_samples = result.samples["L_corr"][:, :, 1, 0]
result.add_derived("rho", rho_samples)

# Now works with all viz functions
viz.plot_density(result, params=["rho"])
print(result.summary(params=["rho"]))
```
- Use `add_derived()` for computed quantities like correlations
- Derived params work with `summary()`, `plot_*()`, and `save()`/`load()`